# Mortality data processing – Cluj-Napoca (SIRUTA 54975)


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

import pandas as pd
from dbfread import DBF

from constants import (
    UAT_ID_CLUJ_NAPOCA, SIRUTA_CODE,
    MORTA_DIR, MORTA_OUTPUT_DIR, DBF_ENCODINGS,
)

OUTPUT_PATH = MORTA_OUTPUT_DIR / f'mortality_UAT_{UAT_ID_CLUJ_NAPOCA}.csv'

print(f'Target SIRUTA code : {SIRUTA_CODE}')
print(f'Mortality directory: {MORTA_DIR.resolve()}')
print(f'Output path        : {OUTPUT_PATH.resolve()}')


In [ ]:
import re

dbf_files = sorted(
    p for p in MORTA_DIR.rglob('*.dbf')
    if re.match(r'MORTA_\d{4}', p.relative_to(MORTA_DIR).parts[0])
)

if not dbf_files:
    raise FileNotFoundError(
        f'No .dbf files found under {MORTA_DIR.resolve()}/MORTA_*/ – '
        'check that the mortality data folders exist.'
    )

print(f'Found {len(dbf_files)} DBF file(s):\n')
for p in dbf_files:
    print(f'  {p}')


In [ ]:
def open_dbf(path, encodings=DBF_ENCODINGS):
    """Try each encoding in order; return the first DBF that doesn't raise."""
    last_err = None
    for enc in encodings:
        try:
            table = DBF(str(path), encoding=enc)
            _ = next(iter(table), None)
            table = DBF(str(path), encoding=enc)
            return table, enc
        except (UnicodeDecodeError, UnicodeWarning) as e:
            last_err = e
            continue
    raise UnicodeDecodeError(
        'all',
        b'',
        0,
        0,
        f'None of the encodings {encodings} could decode {path.name}.  '
        f'Last error: {last_err}',
    )


In [ ]:
sample_table, used_enc = open_dbf(dbf_files[0])

print(f'File    : {dbf_files[0]}')
print(f'Encoding: {used_enc}\n')

print('Fields:')
print(f'  {"Name":20s}  Type  Size  Decimals')
print(f'  {"─"*20}  {"─"*4}  {"─"*4}  {"─"*8}')
for f in sample_table.fields:
    print(f'  {f.name:20s}  {f.type:>4}  {f.length:>4}  {f.decimal_count:>8}')

print('\nFirst 3 records:')
for i, rec in enumerate(sample_table):
    if i >= 3:
        break
    print(dict(rec))


In [ ]:
siruta_col = None
for name in sample_table.field_names:
    if 'SIRUTA' in name.upper():
        siruta_col = name
        break

if siruta_col is None:
    print('No column containing "SIRUTA" found in the DBF schema.')
    print(f'Available columns: {sample_table.field_names}')
    raise KeyError(
        'Could not find a SIRUTA column in the DBF schema.  '
        'Check the field names above and adjust the filter key.'
    )

print(f'SIRUTA column detected: "{siruta_col}"')


In [ ]:
def _extract_year(dbf_path):
    """Extract the year from the MORTA_YYYY ancestor folder."""
    for part in dbf_path.relative_to(MORTA_DIR).parts:
        m = re.match(r'MORTA_(\d{4})', part)
        if m:
            return m.group(1)
    raise ValueError(f'Cannot extract year from path: {dbf_path}')

all_records   = []
total_scanned = 0

for dbf_path in dbf_files:
    year = _extract_year(dbf_path)
    table, enc = open_dbf(dbf_path)
    file_hits = 0

    for rec in table:
        total_scanned += 1
        if int(rec[siruta_col]) == SIRUTA_CODE:
            row = dict(rec)
            row['source_year'] = int(year)
            all_records.append(row)
            file_hits += 1

    print(f'  {dbf_path.name}  (enc={enc})  →  {file_hits:>6,} hits  '
          f'[year {year}]')

print(f'\nTotal records scanned : {total_scanned:,}')
print(f'Total matching records: {len(all_records):,}')


In [ ]:
if not all_records:
    print('No records matched SIRUTA =', SIRUTA_CODE)
    print()

    diag_table, _ = open_dbf(dbf_files[-1])
    unique_siruta = sorted({int(r[siruta_col]) for r in diag_table})
    print(f'Unique SIRUTA values in {dbf_files[-1].name}: '
          f'{len(unique_siruta)} codes')
    print(f'Sample values: {unique_siruta[:20]}')
else:
    df = pd.DataFrame(all_records)

    print(f'DataFrame shape: {df.shape}')
    print(f'\nColumn types:\n{df.dtypes}')
    print(f'\nRecords per year:\n{df["source_year"].value_counts().sort_index().to_string()}')
    print(f'\n{df.head(10)}')


In [ ]:
if not all_records:
    print('Nothing to save – DataFrame is empty.')
else:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f'Saved {len(df):,} rows - {OUTPUT_PATH.resolve()}')
